Fix 1: Verify the immediate problem

In [ ]:
# Can the ingress controller reach headlamp directly?
kubectl exec -n ingress-nginx $(kubectl get pod -n ingress-nginx -o name | head -1) -- \
  curl -s http://10.100.106.147:80 -o /dev/null -w "%{http_code}"

# Check ingress backend endpoints are populated
kubectl get endpoints -n headlamp
kubectl describe ingress -n headlamp

The permanent solution — disable Calico's iptables manipulation for kube-proxy

In [ ]:
kubectl patch felixconfiguration default \
  --type=merge \
  -p '{"spec":{"chainInsertMode":"Append"}}'

In [ ]:
kubectl edit configmap -n kube-system kube-proxy

In [ ]:
mode: "ipvs"
ipvs:
  strictARP: true
  scheduler: "rr"

In [ ]:
kubectl rollout restart daemonset/kube-proxy -n kube-system

Fix 3: Fix headlamp ingress — add the missing annotation

In [ ]:
cat > headlamp-values.yaml << EOF
ingress:
  enabled: true
  ingressClassName: nginx
  annotations:
    nginx.ingress.kubernetes.io/proxy-connect-timeout: "60"
    nginx.ingress.kubernetes.io/proxy-read-timeout: "60"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "60"
  hosts:
    - host: headlamp.voip.local
      paths:
        - path: /
          type: Prefix
EOF

helm upgrade my-headlamp headlamp/headlamp \
  -f headlamp-values.yaml \
  --namespace headlamp

Fix 4: The most likely permanent fix — hairpin/masquerade issue

In [ ]:
kubectl patch felixconfiguration default \
  --type=merge \
  -p '{"spec":{"natPortRange":"32768:65535","bpfConnectTimeLoadBalancingEnabled":false}}'

In [ ]:
kubectl get ippools -o yaml | grep -A3 nat

Quick diagnostic sequence

In [ ]:
# 1. Check ingress sees headlamp endpoints
kubectl get ep -n headlamp

# 2. Check ingress config is generated correctly
kubectl exec -n ingress-nginx daemonset/ingress-nginx-controller -- \
  cat /etc/nginx/nginx.conf | grep -A 10 headlamp

# 3. Check Felix is healthy
kubectl get felixconfiguration default -o yaml

# 4. Test pod-to-pod directly
kubectl run test --rm -it --image=curlimages/curl -- \
  curl http://10.100.106.147:80

In [ ]:
kubectl get installation default -o yaml | grep mtu

In [ ]:
kubectl patch ippool default-ipv4-ippool --type=merge -p \
  '{"spec":{"encapsulation":"None","natOutgoing":true}}'

In [ ]:
kubectl create clusterrolebinding headlamp-admin \
  --clusterrole=cluster-admin \
  --serviceaccount=headlamp:my-headlamp

In [ ]:
kubectl rollout restart deployment/my-headlamp -n headlamp
kubectl rollout status deployment/my-headlamp -n headlamp -w